# Where the wait goes: prefill, decode, a prefix cache and a queue

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/measure-the-latency-your-user-experiences/latency_v1.ipynb)

Companion to https://profrod.ai/articles/measure-the-latency-your-user-experiences. Two small open models (Qwen2.5-Instruct 0.5B and 1.5B), timed phase by phase in a hand-written greedy loop.

Part 1 recomputes the article's numbers from the recorded timings, which came from an Apple M4 Pro GPU. Part 2 reruns a quick version on this runtime. Choose **Runtime > Change runtime type > T4 GPU**. Your absolute numbers will differ from the recorded ones; the shapes should not.

In [ ]:
import json
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/measure-the-latency-your-user-experiences/"
FILES = ["latency_stats_v1.py", "latency_run_v1.py", "data/revisions-v1.json", "results/prefill-v1.jsonl", "results/decode-v1.jsonl",
         "results/cache-v1.jsonl", "results/cache-check-v1.jsonl", "results/queue-v1.jsonl", "receipts/latency-v1.json"]
for name in FILES:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
print("ready")

## Part 1: recompute the article's numbers

In [ ]:
import latency_run_v1 as R

summary = R.summarize()
for model, r in summary["models"].items():
    f = r["prefill"]["fit"]
    print(model, r["device"], r["dtype"])
    print("  prefill median (ms):", {p: round(s * 1000) for p, s in r["prefill"]["medianSeconds"].items()})
    print(f"  fit: a = {f['a'] * 1000:.1f} ms, b = {f['b'] * 1000:.3f} ms/token, c = {f['c'] * 1e9:.2f} ns/token^2")
    for p, d in r["decode"].items():
        print(f"  decode after {p} tokens: {d['medianStepSeconds'] * 1000:.1f} ms per step, {d['tokensPerSecond']:.0f} tokens/s")

In [ ]:
# The prefix cache: 200 trials per condition, p50 and p95 with order-statistic intervals.
for model, r in summary["models"].items():
    print(model)
    for cond, c in r["cache"].items():
        lo, hi = c["p95Interval"] or (float("nan"), float("nan"))
        print(f"  {cond:8s} p50 {c['p50'] * 1000:7.1f} ms   p95 {c['p95'] * 1000:7.1f} ms (95% interval {lo * 1000:.1f} to {hi * 1000:.1f})")
    print("  warm and cold agree on the next token:", r["cacheCheck"]["sameNextToken"], "of", r["cacheCheck"]["questions"])

In [ ]:
# The queue: measured mean time in system against Pollaczek-Khinchine, and Little's law with a sampled L.
for model, r in summary["models"].items():
    print(model)
    for q in r["queue"]["levels"]:
        print(f"  rho {q['rhoMeasured']:.2f}: W {q['meanWait']:.2f} s (formula {q['pkWait']:.2f})   L {q['L']:.2f} vs lambda*W {q['lambdaW']:.2f}")

## Part 2: rerun a quick version here (optional, no key needed)

This runs every experiment on Qwen2.5 0.5B with a tenth of the trials, in a few minutes on a T4, and writes to `mine/`. Keep other work off the runtime while it runs, because timings are sensitive to anything else using the GPU.

In [ ]:
os.makedirs("mine/data", exist_ok=True)
!cp latency_stats_v1.py latency_run_v1.py mine/ && cp data/revisions-v1.json mine/data/
!cd mine && python latency_run_v1.py all --model Qwen/Qwen2.5-0.5B-Instruct --quick && python latency_run_v1.py summarize
mine = json.load(open("mine/results/summary-v1.json"))["models"]["Qwen/Qwen2.5-0.5B-Instruct"]
print("prefill median (ms):", {p: round(s * 1000) for p, s in mine["prefill"]["medianSeconds"].items()})
print("cache p50 (ms):", {c: round(v["p50"] * 1000, 1) for c, v in mine["cache"].items()})